# Preparação do corpus de avaliações

## Objetivo

Consolidar e agrupar as avaliações única tabela para posterior preparação, busca textual e análise de emoções.

Execute as 4 células abaixo em sequencia.

In [8]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import json
from pathlib import Path

import pandas as pd

PASTA_PROJETO = Path("/content/drive/MyDrive/pln_trabalho")
PASTA_RAW = PASTA_PROJETO / "data" / "raw"
PASTA_INTERIM = PASTA_PROJETO / "data" / "interim"

if not PASTA_RAW.is_dir():
    raise FileNotFoundError(f"Pasta não encontrada: {PASTA_RAW}")

ARQUIVO_SAIDA = PASTA_INTERIM / "avaliacoes_consolidadas.csv"

print(f"Origem: {PASTA_RAW}")
print(f"Destino: {ARQUIVO_SAIDA}")

Origem: /content/drive/MyDrive/pln_trabalho/data/raw
Destino: /content/drive/MyDrive/pln_trabalho/data/interim/avaliacoes_consolidadas.csv


In [12]:
arquivos = sorted(PASTA_RAW.glob("*_nota_[1-5].json"))

if len(arquivos) != 50:
    raise ValueError(f"Esperados 50 arquivos; encontrados {len(arquivos)}.")

registros = []

for arquivo in arquivos:
    with arquivo.open("r", encoding="utf-8") as entrada:
        pacote = json.load(entrada)

    registros.extend(pacote["avaliacoes"])

df = pd.DataFrame(registros)

campos_obrigatorios = {"reviewId", "content", "score", "app_id", "app_name"}

if not campos_obrigatorios.issubset(df.columns):
    raise ValueError("Há campos obrigatórios ausentes na tabela.")

if len(df) != 5000:
    raise ValueError(f"Esperadas 5.000 avaliações; encontradas {len(df)}.")

if df["app_id"].nunique() != 10:
    raise ValueError("A tabela não contém exatamente dez aplicativos.")

for campo in ["reviewId", "content"]:
    validos = df[campo].map(
        lambda valor: isinstance(valor, str) and bool(valor.strip())
    )
    if not validos.all():
        raise ValueError(f"Há valores vazios ou inválidos em '{campo}'.")

if df.duplicated(subset=["app_id", "reviewId"]).any():
    raise ValueError("Há IDs repetidos no mesmo aplicativo.")

notas_validas = df["score"].map(
    lambda valor: type(valor) is int and valor in range(1, 6)
)
if not notas_validas.all():
    raise ValueError("Há notas inválidas.")

distribuicao = pd.crosstab(df["app_id"], df["score"]).reindex(
    columns=[1, 2, 3, 4, 5],
    fill_value=0,
)

if not distribuicao.eq(100).all().all():
    raise ValueError("Há quantidades diferentes de 100 por aplicativo e nota.")

print(f"Arquivos lidos: {len(arquivos)}")
print(f"Avaliações: {len(df)}")
print(f"Aplicativos: {df['app_id'].nunique()}")

Arquivos lidos: 50
Avaliações: 5000
Aplicativos: 10


In [11]:
PASTA_INTERIM.mkdir(parents=True, exist_ok=True)

if ARQUIVO_SAIDA.exists():
    raise FileExistsError(
        f"O arquivo já existe: {ARQUIVO_SAIDA}. "
        "O salvamento foi interrompido para evitar sobrescrita."
    )

df.to_csv(
    ARQUIVO_SAIDA,
    index=False,
    encoding="utf-8",
)

print(f"Tabela salva em: {ARQUIVO_SAIDA}")
print(f"Total de avaliações: {len(df)}")

Tabela salva em: /content/drive/MyDrive/pln_trabalho/data/interim/avaliacoes_consolidadas.csv
Total de avaliações: 5000
